# Transformers Docs Copilot — grounded RAG baseline

Answers questions about a **version-pinned** slice of the Hugging Face Transformers docs
and cites the retrieved section. This notebook runs the **untuned 1.5B baseline** that the
QLoRA adapter must beat.

- Repo: public, so no token or login is needed.
- Docs: `huggingface/transformers` @ `v5.24.0` (pinned in `data/raw/docset.json`)
- Model: `Qwen/Qwen2.5-1.5B-Instruct`, 4-bit NF4 on a free Colab T4
- Evaluation: 43 held-out test questions (28 answerable, 15 must abstain)
- **First pass is 2 questions** (`QUICK_LIMIT = 2`) to prove the plumbing. Once that
  works, set it to `0` and rerun the run + inspect cells for the full baseline.
- Full run: about 6–10 minutes after the ~3 GB model download.

**Before running:** `Runtime → Change runtime type → T4 GPU → Save`.

In [1]:
REPO_URL = "https://github.com/titidintsako-bit/transformers-docs-copilot.git"
QUICK_LIMIT = 2          # set to 0 for the full 43-question baseline
SPLIT = "test"

PREFIX = "results/answers_quick" if QUICK_LIMIT else "results/answers_baseline"
LABELS = "results/label_sheet_quick.csv" if QUICK_LIMIT else "results/label_sheet_baseline.csv"
LIMIT_ARG = ["--limit", str(QUICK_LIMIT)] if QUICK_LIMIT else []
print("repo:", REPO_URL)
print("mode:", "quick" if QUICK_LIMIT else "full", "| output:", PREFIX)

repo: https://github.com/titidintsako-bit/transformers-docs-copilot.git
mode: quick | output: results/answers_quick


## 1. Preflight

Checks the GPU, free disk, Python version and that the public repo is reachable. This cell
fails early with a readable message instead of halfway through the run.

In [2]:
import shutil
import subprocess
import sys

import torch

assert torch.cuda.is_available(), "No GPU: Runtime -> Change runtime type -> T4 GPU"
props = torch.cuda.get_device_properties(0)
free_gb = shutil.disk_usage("/content").free / 1e9
assert free_gb > 8, f"Only {free_gb:.1f} GB free in /content; free space and retry"

check = subprocess.run(
    ["git", "ls-remote", REPO_URL, "HEAD"], capture_output=True, text=True, timeout=60
)
assert check.returncode == 0, "Cannot reach the repo. Is it public?\n" + check.stderr[-500:]

print("python:", sys.version.split()[0])
print("torch:", torch.__version__, "|", props.name, f"| {props.total_memory / 1e9:.1f} GB VRAM")
print(f"free disk: {free_gb:.1f} GB")
print("repo reachable:", check.stdout.split()[0][:12])

python: 3.13.16
torch: 2.14.1+cu130 | Tesla T4 | 15.6 GB VRAM
free disk: 120.8 GB
repo reachable: d273bb6b1105


## 2. Get the project

Clones the public repo. Safe to rerun: an existing complete clone is reused, and a broken
partial clone is removed first.

In [3]:
import os
import shutil
import subprocess
from pathlib import Path

PROJECT_DIR = Path("/content/transformers-docs-copilot")
if PROJECT_DIR.exists() and not (PROJECT_DIR / "pyproject.toml").exists():
    shutil.rmtree(PROJECT_DIR)
if not PROJECT_DIR.exists():
    clone = subprocess.run(
        ["git", "clone", "--depth", "1", REPO_URL, str(PROJECT_DIR)],
        capture_output=True, text=True,
    )
    if clone.returncode != 0:
        raise SystemExit("git clone failed:\n" + clone.stderr[-1200:])
os.chdir(PROJECT_DIR)
print("project:", Path.cwd())
print(subprocess.run(["git", "log", "--oneline", "-1"], capture_output=True, text=True).stdout.strip())

project: /content/transformers-docs-copilot
d273bb6 Fail fast in Colab install and corpus checks


## 3. Install pinned runtime dependencies

Installs only what the notebook needs, without touching Colab's PyTorch. If pip asks you to
**restart the runtime**, do it and then `Runtime → Run all` again; the clone and pip cache
survive a restart.

In [4]:
import subprocess
import sys

packages = [
    "PyYAML==6.0.2",
    "transformers==5.24.0",
    "accelerate==1.16.0",
    "safetensors==0.8.0",
    "huggingface-hub==2.4.0",
    "bitsandbytes==0.50.2",
]
subprocess.run([sys.executable, "-m", "pip", "install", "-q", *packages], check=True)

import accelerate
import bitsandbytes
import transformers
import torch

print("transformers", transformers.__version__)
print("torch", torch.__version__, "| accelerate", accelerate.__version__, "| bitsandbytes", bitsandbytes.__version__)


transformers 5.24.0
torch 2.14.1+cu130 | accelerate 1.16.0 | bitsandbytes 0.50.2


## 4. Verify the corpus and rebuild the BM25 index

The committed markdown and chunks are checked against the provenance manifest. `artifacts/`
is not committed, so the index is rebuilt here; this is offline and deterministic.

In [5]:
import subprocess
import sys

commands = [
    ["-m", "src.fetch_docs", "--check"],
    ["-m", "src.eval_questions"],
    ["-m", "src.index"],
]
for args in commands:
    subprocess.run([sys.executable, *args], check=True)


## 5. Run the baseline

For every question: BM25 top-5 → numbered passages → model answer → parser. The run scores
format, abstention, citation validity and gold-section citation hits, then writes a report,
predictions and a prefilled human label sheet.

With `QUICK_LIMIT = 2` this is a plumbing check (`answers_quick`); it does not overwrite the
real baseline.

In [6]:
import subprocess
import sys

cmd = [
    sys.executable, "-m", "src.eval_answers",
    "--split", SPLIT,
    "--model", "base",
    "--quantize", "on",
    "--out-prefix", PREFIX,
    "--labels", LABELS,
    *LIMIT_ARG,
]
print(" ".join(cmd))
subprocess.run(cmd, check=True)

/usr/bin/python3 -m src.eval_answers --split test --model base --quantize on --out-prefix results/answers_quick --labels results/label_sheet_quick.csv --limit 2


CompletedProcess(args=['/usr/bin/python3', '-m', 'src.eval_answers', '--split', 'test', '--model', 'base', '--quantize', 'on', '--out-prefix', 'results/answers_quick', '--labels', 'results/label_sheet_quick.csv', '--limit', '2'], returncode=0)

## 6. Inspect the report

`behavior_accuracy` requires the model itself to choose answer vs abstain; a parser-forced
abstention is a failure, even though the delivered output is safe. `grounded_answer_rate`
is stricter: answered **and** cited a gold section. Low `format_rate` on the untuned model
is the gap QLoRA targets.

In [7]:
import json
from pathlib import Path
from IPython.display import Markdown, display

report = Path(f"{PREFIX}.json")
if not report.exists():
    raise SystemExit(f"No report at {report}; run the previous cell.")
payload = json.loads(report.read_text())
print(json.dumps(payload["overall"], indent=2))
display(Markdown(Path(f"{PREFIX}.md").read_text()))

for row in payload["rows"][:3]:
    print(f"--- {row['qid']} [{row['status']}] {row['question']}")
    print(row["answer"][:500])
    print("citations:", [(source["n"], source["url"]) for source in row["sources"]])

{
  "n": 2,
  "behavior_accuracy": 0.0,
  "answered_rate": 0.0,
  "abstained_rate": 1.0,
  "model_abstain_rate_on_negative": null,
  "delivered_abstention_rate_on_negative": null,
  "over_answer_rate_on_negative": null,
  "parser_save_rate_on_negative": null,
  "false_abstain_rate_on_answerable": 1.0,
  "grounded_answer_rate": 0.0,
  "citation_hit_rate": 0.0,
  "page_citation_hit_rate": 0.0,
  "mean_citation_precision": null,
  "format_rate": 0.0,
  "retrieval_hit_rate": 1.0,
  "repairs": 0,
  "forced_abstain": 2,
  "mean_latency_s": 6.175,
  "median_latency_s": 6.18
}


# Grounded answer baseline (untuned)

- Docs: `huggingface/transformers` @ `v5.24.0` (`eeee11099135`)
- Model: `Qwen/Qwen2.5-1.5B-Instruct` (4-bit NF4), greedy decoding, max_new_tokens=350
- Split: **test**, n=2, k=5 passages
- Contract: 2-6 sentences, `[n]` citations after claims, a `Sources:` block with the exact passage URL,
  or exactly `Insufficient evidence in the retrieved documentation.`
- Automated only: format, abstention, citation validity. Support/factual correctness: see the label sheet.

## Overall

| metric | value |
|---|---|
| n | 2 |
| behaviour accuracy | 0.0 |
| answered | 0.0 |
| grounded answer (answerable) | 0.0 |
| citation hits a gold passage (answerable) | 0.0 |
| citation hits a gold page (answerable) | 0.0 |
| mean citation precision (answered) | None |
| model abstains on negative questions | None |
| delivered abstention on negative questions | None |
| delivered answer on negative questions | None |
| parser saves on negative questions | None |
| false abstain (answerable) | 1.0 |
| exact format | 0.0 |
| section retrieval hit@k (answerable) | 1.0 |
| citation repairs | 0 |
| forced abstentions | 2 |
| median latency (s) | 6.18 |

## By category

| category | n | behaviour | answered | grounded | format | citation precision |
|---|---|---|---|---|---|---|
| answerable | 2 | 0.0 | 0.0 | 0.0 | 0.0 | None |

## By topic

| topic | n | behaviour | answered | citation hit | format |
|---|---|---|---|---|---|
| generation | 2 | 0.0 | 0.0 | 0.0 | 0.0 |

## Over-answered negative questions

None.

## Parser saves on negative questions (model tried to answer)

None.

## False abstentions on answerable questions

- **gs-01** (generation): Which decoding strategy does generate use by default, and how many new tokens can it produce at most when no GenerationConfig is specified?
  - reason: no_valid_citations, retrieved: llm_tutorial#decoding-strategy
- **gs-02** (generation): Why does greedy search break down when generating long sequences?
  - reason: no_valid_citations, retrieved: generation_strategies#greedy-search

## Answered but no cited gold passage

None.

## Format issues

- **gs-01**: no_valid_citations; repairs=[]; invalid=[]
- **gs-02**: no_valid_citations; repairs=[]; invalid=[]

## Method notes

- Retrieval is the untouched BM25 baseline; this run isolates generation behaviour.
- A forced abstention means the parser refused the output (no valid citation, an invented
  URL, or a generation error), not that the model chose to abstain.
- Invalid URLs are replaced with the canonical passage URL and counted as repairs; the
  output can never contain a URL that is absent from the retrieved passages.
- These numbers are a floor for the untuned model, not a RAG quality claim.


--- gs-01 [forced_abstain] Which decoding strategy does generate use by default, and how many new tokens can it produce at most when no GenerationConfig is specified?
Insufficient evidence in the retrieved documentation.
citations: []
--- gs-02 [forced_abstain] Why does greedy search break down when generating long sequences?
Insufficient evidence in the retrieved documentation.
citations: []


## 7. Checkpoint the artifacts to Drive (optional)

Colab disconnects eventually. This copies the report, predictions and label sheet to Drive.
If you decline the Drive prompt, the rest of the notebook is unaffected.

In [ ]:
import shutil
from pathlib import Path

try:
    from google.colab import drive
    drive.mount("/content/drive")
except Exception as exc:
    print("Drive not mounted; skipping checkpoint:", exc)
else:
    dest = Path("/content/drive/MyDrive/transformers-docs-copilot-results")
    dest.mkdir(parents=True, exist_ok=True)
    margin = Path(PREFIX).stem.removeprefix("answers_")
    paths = [
        Path(f"{PREFIX}.md"),
        Path(f"{PREFIX}.json"),
        Path(LABELS),
        Path(PREFIX).parent / f"predictions_{margin}.jsonl",
    ]
    for path in paths:
        if path.exists():
            shutil.copy2(path, dest / path.name)
            print("saved", dest / path.name)

## 8. Next steps

1. If the quick pass worked, set `QUICK_LIMIT = 0` in the first code cell.
2. Rerun the run + inspect + checkpoint cells. You should get `results/answers_baseline.*`.
3. Paste the `overall` JSON or `results/answers_baseline.md` back into the chat.
4. Fill `supported_0_1_2`, `citation_precision_0_1` and `citation_recall_0_1` using
   `docs/EVAL.md`.

Troubleshooting:

- **Restart requested after pip** → restart, then `Run all`; reruns are cheap.
- **No GPU** → change the runtime type to T4.
- **Repo clone 404** → confirm the repo is public and the URL matches.
- **Hugging Face rate limit** → `from huggingface_hub import login; login()` with a read token.
- **CUDA out of memory** → stop the run, `Runtime → Restart session`, and tell me; do not
  silently change the evaluation.